# The Base Classification Model
:label:`sec_classification`

You may have noticed that the implementations from scratch and the concise implementation using framework functionality were quite similar in the case of regression. The same is true for classification. Since many models in this book deal with classification, it is worth adding functionalities to support this setting specifically. This section provides a base class for classification models to simplify future code.


In [1]:
import torch
from d2l import torch as d2l

## The `Classifier` Class


We define the `Classifier` class below. In the `validation_step` we report both the loss value and the classification accuracy on a validation batch. We draw an update for every `num_val_batches` batches. This has the benefit of generating the averaged loss and accuracy on the whole validation data. These average numbers are not exactly correct if the final batch contains fewer examples, but we ignore this minor difference to keep the code simple.


In [2]:
class Classifier(d2l.Module):  #@save
    """The base class of classification models."""
    def validation_step(self, batch):
        Y_hat = self(*batch[:-1])
        self.plot('loss', self.loss(Y_hat, batch[-1]), train=False)
        self.plot('acc', self.accuracy(Y_hat, batch[-1]), train=False)

By default we use a stochastic gradient descent optimizer, operating on minibatches, just as we did in the context of linear regression.


In [3]:
@d2l.add_to_class(d2l.Module)  #@save
def configure_optimizers(self):
    return torch.optim.SGD(self.parameters(), lr=self.lr)

## Accuracy

Given the predicted probability distribution `y_hat`,
we typically choose the class with the highest predicted probability
whenever we must output a hard prediction.
Indeed, many applications require that we make a choice.
For instance, Gmail must categorize an email into "Primary", "Social", "Updates", "Forums", or "Spam".
It might estimate probabilities internally,
but at the end of the day it has to choose one among the classes.

When predictions are consistent with the label class `y`, they are correct.
The classification accuracy is the fraction of all predictions that are correct.
Although it can be difficult to optimize accuracy directly (it is not differentiable),
it is often the performance measure that we care about the most. It is often *the*
relevant quantity in benchmarks. As such, we will nearly always report it when training classifiers.

Accuracy is computed as follows.
First, if `y_hat` is a matrix,
we assume that the second dimension stores prediction scores for each class.
We use `argmax` to obtain the predicted class by the index for the largest entry in each row.
Then we [**compare the predicted class with the ground truth `y` elementwise.**]
Since the equality operator `==` is sensitive to data types,
we convert `y_hat`'s data type to match that of `y`.
The result is a tensor containing entries of 0 (false) and 1 (true).
Taking the sum yields the number of correct predictions.


In [4]:
@d2l.add_to_class(Classifier)  #@save
def accuracy(self, Y_hat, Y, averaged=True):
    """Compute the number of correct predictions."""
    Y_hat = Y_hat.reshape((-1, Y_hat.shape[-1]))
    preds = Y_hat.argmax(axis=1).type(Y.dtype)
    compare = (preds == Y.reshape(-1)).type(torch.float32)
    return compare.mean() if averaged else compare

## Summary

Classification is a sufficiently common problem that it warrants its own convenience functions. Of central importance in classification is the *accuracy* of the classifier. Note that while we often care primarily about accuracy, we train classifiers to optimize a variety of other objectives for statistical and computational reasons. However, regardless of which loss function was minimized during training, it is useful to have a convenience method for assessing the accuracy of our classifier empirically. 


## Exercises

1. Denote by $L_\textrm{v}$ the validation loss, and let $L_\textrm{v}^\textrm{q}$ be its quick and dirty estimate computed by the loss function averaging in this section. Lastly, denote by $l_\textrm{v}^\textrm{b}$ the loss on the last minibatch. Express $L_\textrm{v}$ in terms of $L_\textrm{v}^\textrm{q}$, $l_\textrm{v}^\textrm{b}$, and the sample and minibatch sizes.
1. Show that the quick and dirty estimate $L_\textrm{v}^\textrm{q}$ is unbiased. That is, show that $E[L_\textrm{v}] = E[L_\textrm{v}^\textrm{q}]$. Why would you still want to use $L_\textrm{v}$ instead?
1. Given a multiclass classification loss, denoting by $l(y,y')$ the penalty of estimating $y'$ when we see $y$ and given a probabilty $p(y \mid x)$, formulate the rule for an optimal selection of $y'$. Hint: express the expected loss, using $l$ and $p(y \mid x)$.


[Discussions](https://discuss.d2l.ai/t/6809)



1. Denote by $L_\textrm{v}$ the validation loss, and let $L_\textrm{v}^\textrm{q}$ be its quick and dirty estimate computed by the loss function averaging in this section. Lastly, denote by $l_\textrm{v}^\textrm{b}$ the loss on the last minibatch. Express $L_\textrm{v}$ in terms of $L_\textrm{v}^\textrm{q}$, $l_\textrm{v}^\textrm{b}$, and the sample and minibatch sizes.




To derive the relationship between the true validation loss $L_\textrm{v}$, the quick and dirty estimate $L_\textrm{v}^\textrm{q}$, and the loss on the last minibatch $l_\textrm{v}^\textrm{b}$, let's first understand how the quick and dirty method works.

The quick and dirty estimate $L_\textrm{v}^\textrm{q}$ is computed by averaging the loss values across all validation batches. If all batches have the same size, this would be a correct way to compute the overall loss. However, as mentioned in the notebook, the final batch might contain fewer examples, which introduces a minor discrepancy.

Let's use the following notation:
- $n$ is the total number of validation examples
- $b$ is the minibatch size
- $m = \lceil n/b \rceil$ is the number of minibatches (ceiling of n divided by b)
- The last batch size may be smaller: $b_{\text{last}} = n - (m-1)b$, which is less than or equal to $b$

For the true validation loss $L_\textrm{v}$, we need to weight each example equally:

$L_\textrm{v} = \frac{1}{n} \sum_{i=1}^{n} l_i$

Where $l_i$ is the loss for the $i$-th example.

For the quick and dirty estimate, we're averaging batch-wise losses:

$L_\textrm{v}^\textrm{q} = \frac{1}{m} \sum_{j=1}^{m} l_j^b$

Where $l_j^b$ is the average loss for the $j$-th batch.

Now, we can split the true loss calculation into the first $m-1$ full batches and the potentially smaller last batch:

$L_\textrm{v} = \frac{1}{n} \left[ (m-1)b \cdot \frac{1}{m-1} \sum_{j=1}^{m-1} l_j^b + b_{\text{last}} \cdot l_m^b \right]$

Where $l_m^b = l_\textrm{v}^\textrm{b}$ is the loss on the last minibatch.

Simplifying:

$L_\textrm{v} = \frac{(m-1)b}{n} \cdot \frac{1}{m-1} \sum_{j=1}^{m-1} l_j^b + \frac{b_{\text{last}}}{n} \cdot l_\textrm{v}^\textrm{b}$

The quick and dirty estimate can be expressed as:

$L_\textrm{v}^\textrm{q} = \frac{1}{m} \left[ \sum_{j=1}^{m-1} l_j^b + l_\textrm{v}^\textrm{b} \right]$

From this, we can solve for $\sum_{j=1}^{m-1} l_j^b$:

$\sum_{j=1}^{m-1} l_j^b = m \cdot L_\textrm{v}^\textrm{q} - l_\textrm{v}^\textrm{b}$

Substituting back:

$L_\textrm{v} = \frac{(m-1)b}{n} \cdot \frac{1}{m-1} [m \cdot L_\textrm{v}^\textrm{q} - l_\textrm{v}^\textrm{b}] + \frac{b_{\text{last}}}{n} \cdot l_\textrm{v}^\textrm{b}$

Simplifying:

$L_\textrm{v} = \frac{b}{n} \cdot L_\textrm{v}^\textrm{q} \cdot m - \frac{b}{n} \cdot l_\textrm{v}^\textrm{b} + \frac{b_{\text{last}}}{n} \cdot l_\textrm{v}^\textrm{b}$

$L_\textrm{v} = \frac{bm}{n} \cdot L_\textrm{v}^\textrm{q} + \frac{b_{\text{last}} - b}{n} \cdot l_\textrm{v}^\textrm{b}$

Since $bm = (m-1)b + b_{\text{last}}$ (total examples processed across all batches), we have:

$L_\textrm{v} = \frac{(m-1)b + b_{\text{last}}}{n} \cdot L_\textrm{v}^\textrm{q} + \frac{b_{\text{last}} - b}{n} \cdot l_\textrm{v}^\textrm{b}$

Since $n = (m-1)b + b_{\text{last}}$, we get:

$L_\textrm{v} = L_\textrm{v}^\textrm{q} + \frac{b_{\text{last}} - b}{n} \cdot (l_\textrm{v}^\textrm{b} - L_\textrm{v}^\textrm{q})$

This is our final expression relating the true validation loss to the quick and dirty estimate, accounting for the potentially smaller last batch.

2. Show that the quick and dirty estimate $L_\textrm{v}^\textrm{q}$ is unbiased. That is, show that $E[L_\textrm{v}] = E[L_\textrm{v}^\textrm{q}]$. Why would you still want to use $L_\textrm{v}$ instead?




To show that the quick and dirty estimate $L_\textrm{v}^\textrm{q}$ is unbiased, we need to demonstrate that $E[L_\textrm{v}] = E[L_\textrm{v}^\textrm{q}]$. Let's analyze this step by step.

#### Setting up the problem:

- Let's denote the individual loss for each validation example as $l_1, l_2, ..., l_n$, where $n$ is the total number of validation examples.
- The true validation loss $L_\textrm{v} = \frac{1}{n}\sum_{i=1}^{n}l_i$.
- We have $m$ batches in total, with batch size $b$ for the first $m-1$ batches, and potentially a smaller size $b_{\text{last}}$ for the last batch.
- The quick and dirty estimate $L_\textrm{v}^\textrm{q} = \frac{1}{m}\sum_{j=1}^{m}l_j^b$, where $l_j^b$ is the average loss for batch $j$.

#### Proving unbiasedness:

Taking the expectation of the true validation loss:

$E[L_\textrm{v}] = E\left[\frac{1}{n}\sum_{i=1}^{n}l_i\right] = \frac{1}{n}\sum_{i=1}^{n}E[l_i]$

Now, let's analyze the expectation of the quick and dirty estimate:

$E[L_\textrm{v}^\textrm{q}] = E\left[\frac{1}{m}\sum_{j=1}^{m}l_j^b\right] = \frac{1}{m}\sum_{j=1}^{m}E[l_j^b]$

For each batch $j$ (except possibly the last), $l_j^b = \frac{1}{b}\sum_{i \in \text{batch}(j)} l_i$, which means:

$E[l_j^b] = E\left[\frac{1}{b}\sum_{i \in \text{batch}(j)} l_i\right] = \frac{1}{b}\sum_{i \in \text{batch}(j)} E[l_i]$

The key insight: If the examples are randomly assigned to batches and all examples come from the same distribution (which is a standard assumption in validation), then $E[l_i]$ is the same for all examples, let's call it $\mu$.

Therefore:
- $E[L_\textrm{v}] = \frac{1}{n}\sum_{i=1}^{n}\mu = \mu$
- For batches 1 to $m-1$: $E[l_j^b] = \frac{1}{b}\sum_{i \in \text{batch}(j)} \mu = \mu$
- For the last batch: $E[l_m^b] = \frac{1}{b_{\text{last}}}\sum_{i \in \text{batch}(m)} \mu = \mu$

Thus:
$E[L_\textrm{v}^\textrm{q}] = \frac{1}{m}\sum_{j=1}^{m}\mu = \mu = E[L_\textrm{v}]$

This proves that the quick and dirty estimate is unbiased.

#### Why still use $L_\textrm{v}$ instead?

Although $L_\textrm{v}^\textrm{q}$ is unbiased, there are several reasons to prefer $L_\textrm{v}$:

1. **Variance Reduction**: While the means are the same, $L_\textrm{v}^\textrm{q}$ typically has higher variance than $L_\textrm{v}$, especially if the last batch is significantly smaller than the others. The true validation loss gives equal weight to each example, reducing this source of variance.

2. **Consistency in Reporting**: When comparing models or different validation runs, using the same weighting scheme for all examples ensures consistency in reported metrics.

3. **Interpretability**: $L_\textrm{v}$ has a clear interpretation as the average loss per example, which is more intuitive than an average of batch losses where batches might have different sizes.

4. **Statistical Efficiency**: When working with limited validation data, $L_\textrm{v}$ makes optimal use of all examples by weighting them equally, which can lead to more reliable model selection decisions.

5. **Edge Cases**: If the last batch is very small (e.g., just one example), its contribution to $L_\textrm{v}^\textrm{q}$ would be disproportionately large relative to its actual importance in the dataset.

In practice, the difference between $L_\textrm{v}$ and $L_\textrm{v}^\textrm{q}$ is often small, especially with large datasets and when the last batch isn't much smaller than the others. This is why the notebook mentions it as a "minor difference" that's often ignored for simplicity. However, for rigorous evaluation, especially in research settings, using the properly weighted $L_\textrm{v}$ is preferable.

3. Given a multiclass classification loss, denoting by $l(y,y')$ the penalty of estimating $y'$ when we see $y$ and given a probabilty $p(y \mid x)$, formulate the rule for an optimal selection of $y'$. Hint: express the expected loss, using $l$ and $p(y \mid x)$.




To determine the optimal prediction rule for multiclass classification, we need to find the class label $y'$ that minimizes the expected loss for a given input $x$.

#### Setting up the Expected Loss

Let's define:
- $l(y, y')$: the loss or penalty incurred when the true class is $y$ but we predict $y'$
- $p(y \mid x)$: the conditional probability of class $y$ given input $x$

For a given input $x$, the expected loss when predicting class $y'$ is:

$$\mathcal{L}(y' \mid x) = \mathbb{E}_y[l(y, y') \mid x] = \sum_{y} p(y \mid x) \cdot l(y, y')$$

The optimal prediction $y^*$ is the one that minimizes this expected loss:

$$y^* = \arg\min_{y'} \mathcal{L}(y' \mid x) = \arg\min_{y'} \sum_{y} p(y \mid x) \cdot l(y, y')$$

#### Specific Cases

1. **0-1 Loss**: If $l(y, y') = \mathbb{1}(y \neq y')$ (i.e., 0 if correct, 1 if wrong), then:

   $$\mathcal{L}(y' \mid x) = \sum_{y} p(y \mid x) \cdot \mathbb{1}(y \neq y') = \sum_{y \neq y'} p(y \mid x) = 1 - p(y' \mid x)$$

   To minimize this, we select the class with the highest probability:
   
   $$y^* = \arg\max_{y'} p(y' \mid x)$$
   
   This is the standard Maximum A Posteriori (MAP) decision rule.

2. **Asymmetric Loss**: If different types of misclassifications have different costs, we use the full loss function:

   $$y^* = \arg\min_{y'} \sum_{y} p(y \mid x) \cdot l(y, y')$$

#### Intuitive Explanation

This result has an elegant interpretation: the optimal prediction balances the probability of each possible true class with the specific cost of mistaking it for our prediction.

For example, in a medical diagnosis where false negatives (missing a disease) are more costly than false positives (unnecessary treatment), the loss function would weight these errors differently. Even if the probability of disease is lower than no disease, the higher cost of missing it might lead us to predict the disease is present.

#### General Form of the Decision Rule

More generally, given any loss function $l(y, y')$ and the posterior distribution $p(y \mid x)$, the optimal decision rule is to calculate the expected loss for each possible prediction $y'$ and choose the one with the minimum expected loss.

This result is a cornerstone of statistical decision theory and extends beyond classification to any decision-making problem under uncertainty where we can quantify probabilities and losses.

## Honestly I have no clue about the above answers